# 03a · Anotación en CVAT: lote estratificado (pre-anotación YOLO26 + revisión humana)

## Motivación / Objetivos

Si las etiquetas de entrenamiento se generan con el mismo detector preentrenado que luego se ajusta, el mAP solo mide cuánto se parece el modelo a su "profesor", no a la realidad (entrenamiento circular). Aquí las cajas las propone YOLO26, pero **un humano las revisa y corrige en CVAT**, y solo las etiquetas revisadas pasan a entrenamiento y test.

Objetivos:
- Crear lotes de imágenes representativos de cámaras, horas, tipos de día e iluminación.
- Pre-anotarlos con YOLO26 para acelerar la revisión.
- Fijar un protocolo de anotación común.
- Exportar las etiquetas revisadas en formato YOLO.

**Objetivos del TFM relacionados:** OE-3 (modelo que detecte peatones, vehículos y semáforos) y OE-6 (referencia humana para medir el rendimiento).

**Requisitos:** Docker Desktop en ejecución y CVAT arrancado:

```powershell
.\cvat\cvat.ps1 instalar   # solo la primera vez (clona CVAT v2.78.0, arranca y crea el usuario admin)
.\cvat\cvat.ps1 arrancar   # sesiones posteriores
```

## Método

```
02a/02b ──► data/malaga/fases/{frames,weather}.parquet  (fotogramas válidos + contexto)
   │
03a (este) ──► 1. muestreo estratificado (cámara × franja × tipo de día × iluminación)
   │           2. tarea CVAT + pre-anotación automática YOLO26 (cvat_sdk.auto_annotation)
   │           3. revisión humana en http://localhost:8080
   │           4. exportación ──► data/malaga/anotacion/<lote>/labels/*.txt
   │                              data/malaga/anotacion/<lote>/indice.csv
   ▼
04 ──► lee todos los anotacion/*/indice.csv (solo estado = revisada)
```

Las clases están en `src/deteccion.py` (`CLASS_NAMES`), compartidas con `04`. La lógica de CVAT está en `src/cvat_lotes.py`, compartida con `03b` y `03c`. La función de pre-anotación está en `src/cvat_yolo26.py`.

1. Configuración.
2. Conexión con CVAT.
3. Selección estratificada de fotogramas.
4. Proyecto y tarea en CVAT.
5. Pre-anotación automática con YOLO26.
6. Revisión humana en CVAT (protocolo de anotación).
7. Exportación de las etiquetas revisadas.
8. Resumen del lote.

### 1. Configuración

In [ ]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from config import load_config
from cvat_lotes import (annotation_counts, connect, create_lote, export_lote, get_or_create_project,
                        images_in_lotes, jobs_status, load_task, lote_dir, pre_annotate, task_url)
from deteccion import CLASS_NAMES
from pipeline import image_path, load_frames, only_service_cameras, stratified_selection

CFG = load_config()

# Lote de anotación (config.yaml -> cvat): cada lote nuevo excluye las imágenes de lotes anteriores
LOTE = CFG["cvat"]["lote"]
N_IMAGENES_LOTE = CFG["cvat"]["n_imagenes_lote"]
MAX_POR_CAMARA = CFG["cvat"]["max_por_camara"]     # evita que unas pocas cámaras dominen el lote
SEGMENT_SIZE = CFG["cvat"]["segment_size"]         # imágenes por job de CVAT (unidad de revisión)
RANDOM_STATE = CFG["entrenamiento"]["semilla"]
MIN_GAP_S = CFG["cvat"]["separacion_minima_min"] * 60   # entre imágenes de la misma cámara

# Pre-anotación YOLO26 (mismos parámetros que 02c_preproceso_deteccion_yolo26)
PRE_WEIGHTS = CFG["deteccion"]["modelo"]
PRE_CONF = CFG["deteccion"]["conf"]
PRE_IMGSZ = CFG["deteccion"]["imgsz"]

print("Clases:", CLASS_NAMES)
print("Lote:", lote_dir(LOTE))

### 2. Conexión con CVAT

La versión del servidor debe coincidir con `cvat-sdk` (fijada en `requirements.txt` y en `cvat/cvat.ps1`).

In [ ]:
# Credenciales de cvat/.env (generadas por cvat/cvat.ps1, excluidas de git)
client = connect()
project = get_or_create_project(client)
print(f"Conectado a CVAT | proyecto {project.id}: {project.name}")

### 3. Selección estratificada de fotogramas

Se parte de los fotogramas válidos de la fase `frames` (unida con `weather` si existe), solo de las 24 cámaras del servicio. El lote se reparte en *round-robin* (`pipeline.stratified_selection`) entre estratos **cámara × franja horaria × tipo de día × iluminación**, con un máximo por cámara y al menos `separacion_minima_min` (10 min) entre dos imágenes de la misma cámara, para evitar casi-duplicados. Así el lote cubre día y noche, laborables y fines de semana y el mayor número de cámaras posible.

Se excluyen las imágenes de lotes anteriores. Las imágenes se copian con el nombre único `<cámara>_<timestamp>.jpg`, porque los timestamps se repiten entre cámaras.

In [ ]:
# Solo las 24 cámaras del servicio (las demás tienen 1-2 instantáneas sueltas)
frames = only_service_cameras(load_frames("weather"))
frames = frames[frames["path"].map(lambda p: image_path(p).exists())]
frames["image"] = frames["camera"] + "_" + frames["image"]

# Imágenes ya incluidas en otros lotes
already = images_in_lotes(exclude=LOTE)
candidates = frames[~frames["image"].isin(already)]
print(f"Fotogramas disponibles: {len(frames)} | ya en otros lotes: {len(already)} | candidatos: {len(candidates)}")

strata = [c for c in ["camera", "time_slot_2h", "day_type", "illumination"] if c in candidates.columns]
lote = stratified_selection(candidates, strata, N_IMAGENES_LOTE, MAX_POR_CAMARA, RANDOM_STATE, min_gap_s=MIN_GAP_S)

print(f"Imágenes seleccionadas: {len(lote)} de {lote['camera'].nunique()} cámaras")
for col in ["time_slot_2h", "day_type", "illumination", "weather"]:
    if col in lote.columns:
        print(f"\n{col}:\n{lote[col].value_counts().sort_index().to_string()}")

In [ ]:
# Crea la carpeta del lote (imágenes + indice.csv) y su tarea en CVAT (src/cvat_lotes.py)
if (lote_dir(LOTE) / "indice.csv").exists():
    print(f"El lote {LOTE} ya existe; se reutiliza. Cambia cvat.lote en config.yaml para crear otro.")
else:
    task = create_lote(client, project, LOTE, lote, SEGMENT_SIZE)
    print(f"Lote creado: {len(lote)} imágenes en {lote_dir(LOTE)} | tarea {task.id}")

### 4. Proyecto y tarea en CVAT

El proyecto define las etiquetas (iguales a `CLASS_NAMES`). Cada lote es una tarea dividida en jobs de `SEGMENT_SIZE` imágenes.

In [ ]:
task = load_task(client, LOTE)
print(f"Tarea {task.id} ({task.size} imágenes) | abrir en el navegador: {task_url(task)}")
display(jobs_status(task))

### 5. Pre-anotación automática con YOLO26

`cvat_sdk.auto_annotation.annotate_task` descarga cada imagen de la tarea, ejecuta `Yolo26PreAnnotator` (pesos COCO) en local y sube las cajas a CVAT con `source = auto`:

- `pedestrian` (clase COCO `person`), `car`, `bus`, `truck`, `motorcycle`: clase COCO directa.
- Semáforos: color HSV por caja → `traffic_light_red/yellow/green`, o `traffic_light_unknown`.

Las pre-anotaciones son solo un punto de partida y se revisan todas en el paso 6. La pre-anotación sustituye todas las cajas de la tarea, así que **se niega a ejecutarse si la tarea ya tiene cajas manuales o semiautomáticas**.

In [ ]:
pre_annotate(client, task, PRE_WEIGHTS, PRE_CONF, PRE_IMGSZ)
print(f"Pre-anotación completada. Revisar en {task_url(task)}")

### 6. Revisión humana en CVAT (protocolo de anotación)

Abrir la tarea en CVAT y revisar **cada job**. Criterios comunes para que las etiquetas sean consistentes (documentarlos en la memoria):

| Situación | Acción |
|---|---|
| Caja correcta | Mantener; ajustar los bordes al objeto visible. |
| Falso positivo | Borrar. |
| Objeto no detectado | Dibujar una caja nueva (vehículo o persona visible de más de ~8 px de alto). |
| Semáforo **peatonal** | Cambiar la etiqueta a `pedestrian_light_red` / `pedestrian_light_green` (la pre-anotación no los distingue). |
| Semáforo de lado, de espaldas o sin color distinguible | `traffic_light_unknown`. |
| Color mal clasificado (reflejos, halos nocturnos) | Corregir la etiqueta de color. |
| Objeto ocluido más de un 70 % o cortado en el borde e irreconocible | No anotar. |
| Grupo compacto de personas | Una caja por persona distinguible. |
| Coche (`car`): atributo `estado` | `esperando` si está detenido ante el semáforo en el carril de espera; `aparcado` si está estacionado; `circulando` si está en marcha; `desconocido` si no se puede saber. Sirve para validar `06_clasificacion_estado_vehiculos`; basta con marcarlo en las imágenes de las cámaras de referencia (lote de validación de `03c`). |

Al terminar cada job: **Menu → Finish the job** (estado `completed`). La exportación incluye **solo los jobs terminados**, así que se puede exportar y entrenar con lo ya revisado sin esperar a terminar el lote. Para priorizar las imágenes que más equilibran las clases, ver `03b_anotacion_lote_prioritario`.

In [ ]:
estado_jobs = jobs_status(task)
display(estado_jobs)
terminados = (estado_jobs["estado"] == "completed").sum()
print(f"Jobs terminados: {terminados} de {len(estado_jobs)}. La exportación incluye solo los terminados.")

### 7. Exportación de las etiquetas revisadas

Se exporta en formato YOLO y se reasignan los índices de clase **por nombre** a `CLASS_NAMES` (`anotacion.remap_lines`), de modo que el orden de las etiquetas en CVAT no importa. Solo las imágenes de **jobs terminados** pasan a `revisada`. Una imagen revisada sin cajas genera un `.txt` vacío (ejemplo negativo válido). La exportación se puede repetir a medida que se terminan jobs.

In [ ]:
indice = export_lote(client, task, LOTE)
revisadas = indice[indice["estado"] == "revisada"]
print(f"Etiquetas escritas en {lote_dir(LOTE) / 'labels'}: {len(revisadas)} imágenes revisadas, "
      f"{int(revisadas['n_cajas'].sum())} cajas | pendientes: {(indice['estado'] == 'pendiente').sum()}")

### 8. Resumen del lote

Distribución de cajas por clase y cámara. Las clases con muy pocas instancias (p. ej. `traffic_light_yellow`) indican qué situaciones debe sobremuestrear el siguiente lote.

In [ ]:
labels_dir = lote_dir(LOTE) / "labels"
rows = []
for label_file in labels_dir.glob("*.txt"):
    camera = label_file.stem.split("_", 1)[0]
    for line in label_file.read_text(encoding="utf-8").splitlines():
        rows.append({"camera": camera, "clase": CLASS_NAMES[int(line.split()[0])]})

boxes = pd.DataFrame(rows, columns=["camera", "clase"])
print("Cajas revisadas por clase:")
display(boxes["clase"].value_counts().reindex(CLASS_NAMES, fill_value=0).to_frame("cajas"))
print(f"Imágenes revisadas sin ningún objeto: {(revisadas['n_cajas'] == 0).sum()}")
display(pd.crosstab(boxes["camera"], boxes["clase"]))

print("Cajas en CVAT por origen (auto = pre-anotación sin tocar):")
display(annotation_counts(client, task))

## Salida / Resultados

| Salida | Contenido |
|---|---|
| `data/malaga/anotacion/<lote>/images/` | Copia de las imágenes del lote (no versionada). |
| `data/malaga/anotacion/<lote>/indice.csv` | Imagen, cámara, contexto y `estado` (`pendiente` / `revisada`). |
| `data/malaga/anotacion/<lote>/labels/*.txt` | Etiquetas YOLO revisadas (versionadas: es trabajo humano). |

**Estado (17/10/2026):** `lote_01` tiene 300 imágenes (tarea 1, 6 jobs de 50), todas `pendiente` en `indice.csv` porque ningún job está terminado. El job 1 está en curso, con cambios en 22 de sus primeras 24 imágenes y 104 cajas humanas.

## Observaciones

- El notebook está guardado sin salidas. Hay que volver a ejecutarlo para registrar el resumen del lote.
- Objetivo de anotación: un primer hito de ~600 imágenes revisadas para entrenar y trazar la curva de aprendizaje, y de 1.500 a 2.000 si la curva lo justifica.
- Se recomienda que una segunda persona anote el ~10 % de las imágenes, para medir el acuerdo entre anotadores (IoU y F1 por clase).
- Siguiente fase: `04_entrenamiento_yolo26`, que lee todos los `data/malaga/anotacion/*/indice.csv` con `estado = revisada` y entrena con validación cruzada por cámara.